# N-Gram Language Models
In this exercise, we will use n-gram language models to predict the probability of text, and generate it.

In [46]:
import nltk

from nltk.corpus import gutenberg

First, we load Jane Austen's Emma from NLTK's gutenberg corpus that we also used in a previous exercise. Tokenize and lowercase this text such that we have a list of words.

In [47]:
raw_text = gutenberg.raw('austen-emma.txt')

# words = [w.lower() for w in nltk.word_tokenize(raw_text)]
words = [w.lower() for w in raw_text.split()]

print(len(words))
# print(len(words2))

158167


Write an n-gram language model class that takes the word list and a parameter `n` as inputs, where `n` is a positive integer larger than 1 that determines the `n` of the n-gram LM. The LM should build a dictionary of n-gram counts from the word list.

In [48]:
from collections import defaultdict

class NGramLanguageModel:
    
    def __init__(self, words, n):
        self.n = n
        assert n > 1, "n needs to be a positive integer > 1"
        assert n <= len(words), "n can't be larger than the number of words"
        
        self.counts = defaultdict(int)
        for i in range(len(words) - n + 1):
            ngram = tuple(words[i : i+n])
            self.counts[ngram] += 1
            
            ngram_minus_one = ngram[:-1]
            self.counts[ngram_minus_one] += 1
        
        ngram_minus_one = ngram[1:]
        self.counts[ngram_minus_one] += 1

Now we "train" the n-gram LM by building the n-gram counts of the Emma novel. Use a low `n` (i.e. 2 or 3).

In [49]:
ngrams = NGramLanguageModel(words, n=3)

Let's add a method `log_probability` to the n-gram LM class that computes the probability of an input string. Since multiplying many probabilities (<= 1) results in very small numbers that can underflow, we sum the log probabilities instead.

In [50]:
import math

def log_probability(self, input_string):
    """ Returns the log-probability of the input string."""
    # example: [the, cat, sat, on, the, mat]
    # p(example) = p(sat | the cat) * p(on | cat sat) * p(the | sat on) * p(mat | on the)
    input_words = [w.lower() for w in nltk.word_tokenize(input_string)]
    log_sum = 0
    counter = 0
    for i in range(len(input_words) - self.n + 1):
        ngram = tuple(input_words[i : i+self.n])
        if ngram in self.counts:
            prob = self.counts[ngram] / self.counts[ngram[:-1]]
            log_sum += math.log(prob)
            counter += 1    
    return log_sum / counter

NGramLanguageModel.log_probability = log_probability

Shorter texts will have higher log probability than longer texts, so we need to normalize it by the number of words in the input string.

In [51]:
print(ngrams.log_probability('There is a house in New Orleans.'))

-2.159484249353372


Lets predict the probabilities of two novels under our trained model: Jane Austen's *Sense and Sensibility* (`austen-sense.txt`) and Shakespeare's *Hamlet* (`shakespeare-hamlet.txt`).
- What do you expect will happen?
- What do you observe?

In [52]:
austen_sense = gutenberg.raw('austen-sense.txt')
shakespeare_hamlet = gutenberg.raw('shakespeare-hamlet.txt')
print(ngrams.log_probability('There is a house in New Orleans.'))
print(ngrams.log_probability(austen_sense))
print(ngrams.log_probability(shakespeare_hamlet))

-2.159484249353372
-2.2169294620028315
-2.396871324477108


How many n-grams are known in each input?

In [53]:
def known_ngrams(self, input_string):
    counter = 0
    words = [w.lower() for w in input_string.split()]
    for i in range(len(words) - self.n + 1):
        ngram = tuple(words[i:i+self.n])
        if ngram in self.counts:
            counter += 1
            
    return counter, len(words) - self.n + 1

NGramLanguageModel.known_ngrams = known_ngrams

In [56]:
known, total = ngrams.known_ngrams(raw_text)
print(f'Known n-grams in Emma: {known/total:.2%} ({known}/{total})')
known, total = ngrams.known_ngrams(austen_sense)
print(f'Known n-grams in Sense and Sensibility: {known/total:.2%} ({known}/{total})')
known, total = ngrams.known_ngrams(shakespeare_hamlet)
print(f'Known n-grams in Hamlet: {known/total:.2%} ({known}/{total})')

Known n-grams in Emma: 100.00% (158165/158165)
Known n-grams in Sense and Sensibility: 15.10% (17925/118673)
Known n-grams in Hamlet: 3.06% (906/29603)


Let's add a method `generate` that takes the start of a sentence ("prompt") and a number of words to generate, then continues our prompt.

In [57]:
def generate(self, prompt, num_words=10):
    """ Continues a text starting with `prompt` for the `num_words` next words. """
    words = [w.lower() for w in prompt.split()]
    for i in range(num_words):
        prefix = tuple(words[-(self.n - 1):])
        if prefix not in self.counts:
            words.append('[END]')
            break
        next_word_dist = {}
        for ngram in self.counts:
            if len(ngram) == self.n and ngram[:-1] == prefix:
                next_word_dist[ngram] = self.counts[ngram] / self.counts[ngram[:-1]]
        
        print(prefix, next_word_dist)
        best_ngram, prob = max(next_word_dist.items(), key=lambda x: x[1])
        print(best_ngram, prob)
        words.append(best_ngram[-1])
        
    return ' '.join(words)

NGramLanguageModel.generate = generate

Play around with a few different prompts.

In [58]:
ngrams.generate('I went for a walk')

('a', 'walk') {('a', 'walk', 'with'): 0.3333333333333333, ('a', 'walk', 'before'): 0.3333333333333333, ('a', 'walk', 'in'): 0.3333333333333333}
('a', 'walk', 'with') 0.3333333333333333
('walk', 'with') {('walk', 'with', 'the'): 0.25, ('walk', 'with', 'his'): 0.25, ('walk', 'with', 'her,'): 0.5}
('walk', 'with', 'her,') 0.5
('with', 'her,') {('with', 'her,', 'but'): 0.21052631578947367, ('with', 'her,', 'as'): 0.05263157894736842, ('with', 'her,', 'instead'): 0.05263157894736842, ('with', 'her,', 'or'): 0.05263157894736842, ('with', 'her,', 'and'): 0.15789473684210525, ('with', 'her,', 'it'): 0.05263157894736842, ('with', 'her,', 'of'): 0.05263157894736842, ('with', 'her,', 'that'): 0.10526315789473684, ('with', 'her,', 'with'): 0.05263157894736842, ('with', 'her,', 'how'): 0.05263157894736842, ('with', 'her,', 'after'): 0.05263157894736842, ('with', 'her,', 'she'): 0.05263157894736842, ('with', 'her,', 'i'): 0.05263157894736842}
('with', 'her,', 'but') 0.21052631578947367
('her,', 'but

'i went for a walk with her, but emma could not be in a very'

In [59]:
ngrams.generate('I went for a walk', num_words=30)

('a', 'walk') {('a', 'walk', 'with'): 0.3333333333333333, ('a', 'walk', 'before'): 0.3333333333333333, ('a', 'walk', 'in'): 0.3333333333333333}
('a', 'walk', 'with') 0.3333333333333333
('walk', 'with') {('walk', 'with', 'the'): 0.25, ('walk', 'with', 'his'): 0.25, ('walk', 'with', 'her,'): 0.5}
('walk', 'with', 'her,') 0.5
('with', 'her,') {('with', 'her,', 'but'): 0.21052631578947367, ('with', 'her,', 'as'): 0.05263157894736842, ('with', 'her,', 'instead'): 0.05263157894736842, ('with', 'her,', 'or'): 0.05263157894736842, ('with', 'her,', 'and'): 0.15789473684210525, ('with', 'her,', 'it'): 0.05263157894736842, ('with', 'her,', 'of'): 0.05263157894736842, ('with', 'her,', 'that'): 0.10526315789473684, ('with', 'her,', 'with'): 0.05263157894736842, ('with', 'her,', 'how'): 0.05263157894736842, ('with', 'her,', 'after'): 0.05263157894736842, ('with', 'her,', 'she'): 0.05263157894736842, ('with', 'her,', 'i'): 0.05263157894736842}
('with', 'her,', 'but') 0.21052631578947367
('her,', 'but

'i went for a walk with her, but emma could not be in a very good sort of thing that is quite a different sort of thing that is quite a different sort of thing'